# 0.Instalaciones

In [54]:
#Librerias para modelos LLM
!pip install -U transformers accelerate bitsandbytes sentencepiece

#Librerias para embeddings, vector search y datasets
!pip install faiss-cpu sentence-transformers datasets

#Libreria rankBM25
!pip install rank-bm25

# 1.Dataset Externo

Se utiliza el dataset CNN / DailyMail, que contiene articulos reales de noticias. De donde se seleccionaran 200 documentos para evitar sobrecargar la memoria de Colab.

Cada documento corresponde a un articulo periodistico real.

In [55]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0", split="train")

In [56]:
texts = []
for row in dataset:
    texts.append(row["article"])
    if len(texts) == 200:
        break

In [57]:
print("Docs:", len(texts))

Docs: 200


# 2.Chunking con overlap

Dado que los modelos de embeddings y los LLM funcionan mejor con fragmentos de texto pequeños. Por ello dividimos cada documento en chunks.

Usaremos chunk size de 400 para mantener equilibrio entre contexto y precisión en retrieval.

y overlap de 100 para mantener continuidad semantica.

In [58]:
def chunk_text(text, size=400, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start+size]
        chunks.append(chunk)
        start += size - overlap

    return chunks

In [59]:
#Aplicamos el chunking:
chunks = []

for t in texts:
    chunks.extend(chunk_text(t))

print("Total chunks:", len(chunks))

Total chunks: 2485


In [60]:
#Revisamos un chunck
print(chunks[0])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his ca


Creamos un indice BM25

In [61]:
# --------------------------------
# BM25 Index (Lexical Retrieval)
# --------------------------------

from rank_bm25 import BM25Okapi

# Tokenizamos los chunks
tokenized_chunks = [c.split() for c in chunks]

# Creamos el índice BM25
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index creado")

BM25 index creado


# 3.Embeddings

Los embeddings convierten texto en vectores numéricos que representan su significado semántico.

Para ello utilizaremos el modelo all-MiniLM-L6-v2, ya que es rápido, ligero y ofrece una buena calidad semántica.

In [62]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

# 4.FAISS con HNSW

Para la indexación vectorial se utilizó FAISS con el índice HNSW, un algoritmo basado en grafos jerarquicos que permite realizar búsquedas de vecinos más cercanos de forma eficiente en grandes colecciones de vectores

In [63]:
import faiss

dim = embeddings.shape[1]

# normalizar vectores
faiss.normalize_L2(embeddings)

# crear índice HNSW
index = faiss.IndexHNSWFlat(dim, 32) #Nro de conexiones entre nodos

# agregar embeddings al índice
index.add(embeddings)

print("Vectores indexados:", index.ntotal)

Vectores indexados: 2485


# 5.RETRIEVAL

Se recuperan los 5 chunks más relevantes (k=5) mediante búsqueda vectorial en FAISS, los cuales se utilizan como contexto para la generación de respuestas

In [64]:
def retrieve(query, k=5):

    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)

    D, I = index.search(q_emb, k)

    return [chunks[i] for i in I[0]]

El retrieval busca los k chunks mas similares a la pregunta. Estos documentos se usarán como contexto para el LLM.

In [65]:
#Funcion Retrieve BM25
def retrieve_bm25(query, k=5):

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top_idx = np.argsort(scores)[::-1][:k]

    return [chunks[i] for i in top_idx]

Se utiliza **recuperación híbrida (BM25 + FAISS)** para encontrar documentos relevantes. **FAISS** realiza búsqueda semántica mediante *embeddings*, permitiendo encontrar textos con significado similar a la consulta, mientras que **BM25** busca coincidencias exactas de palabras clave. La combinación de ambos métodos mejora la recuperación de documentos en el sistema **RAG**. Luego, un **Reranker (cross-encoder)** selecciona los documentos más relevantes para enviarlos al **LLM** como contexto.

In [66]:
#Funcion hybrid retrieve:
def hybrid_retrieve(query, k=10):

    # Retrieval semántico (FAISS)
    docs_faiss = retrieve(query, k)

    # Retrieval léxico (BM25)
    docs_bm25 = retrieve_bm25(query, k)

    # Combinamos resultados
    combined = list(set(docs_faiss + docs_bm25))

    return combined

In [67]:
#Probamos
query = "What is climate change?"

#docs = retrieve(query)
# Hybrid Retrieval: combina FAISS (semántico) + BM25 (léxico)
# para mejorar la recuperación de documentos relevantes
docs = hybrid_retrieve(query, k=10)

for d in docs:
    print(d[:200])
    print("-----")

nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming. Many global warming experts say the phenomenon, if unchecked, is
-----
ts. "For the last 30 years, there's no way there's anything natural that can explain it," Stephen Schneider, a professor of environmental studies at Stanford University in California, said. "A vast bu
-----
acted as though the rules -- even those of geography and climate -- did not apply to him. But when you're Wright, you're right. In 1935, department store magnate Stanley Marcus (of Neiman-Marcus fame)
-----
here are those who do not share his view, and among the skeptics is Richard Lindzen, a professor of atmospheric sciences at the Massachusetts Institute of Technology. "We've suddenly taken to reading 
-----
, Fallingwater, Taliesin West, the Guggenheim, and countless other design benchmarks, Frank Lloyd Wright is arguably the genius of 20th-century architecture. And, boy, did 

# 6.RERANKER CROSS ENCODER


In [68]:
!pip install sentence-transformers

Se implementó un reranker basado en un cross-encoder, que evalúa la relevancia entre la consulta y cada documento recuperado por FAISS para mejorar la calidad del contexto enviado al modelo generativo.

In [69]:
#Cargar reranker CROSS ENCODER
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [70]:
#Funcion Reranking:
def rerank(query, docs, top_k=3):

    pairs = [[query, doc] for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, score in ranked[:top_k]]

In [71]:
#docs = retrieve(query, k=10)
docs = hybrid_retrieve(query, k=10)

reranked_docs = rerank(query, docs, top_k=5)

context = "\n".join(reranked_docs)

# 7.LLM

Se utiliza el modelo Qwen 1.5 1.8B Chat, un modelo conversacional con buena capacidad de razonamiento y relativamente ligero en comparación con modelos más grandes. Además, se aplica cuantización a 4 bits con el objetivo de reducir el uso de memoria y permitir una ejecución más eficiente.

In [72]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen1.5-1.8B-Chat"

bnb = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5504, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5504, bias=False)
          (down_proj): Linear4bit(in_features=5504, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm(

Prompt + generación

In [73]:
def ask_llm(context, question):

    messages = [
        {"role": "system", "content": "Answer ONLY using provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=200)

    return tokenizer.decode(out[0], skip_special_tokens=True)


In [74]:
#Probamos
query = "What causes climate change?"

#docs = retrieve(query, k=10)
docs = hybrid_retrieve(query, k=10)

reranked_docs = rerank(query, docs)

context = "\n".join(reranked_docs)

answer = ask_llm(context, query)

print(answer)

system
Answer ONLY using provided context.
user
Context:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming. Many global warming experts say the phenomenon, if unchecked, is capable of altering the world's climate and geography. In the worst-case scenario, experts say oceans could rise to overwhelming and catastrophic levels, flooding cities and altering seashores. Other
 be significant because it was going to be an example of climate change." "The other thing was that it meant it was really happening. It wasn't a joke. It wasn't just statistics. It was really happening." He calls his discovery Warming Island. Many climatologists and scientists say arctic ice melt and other changes in the Earth's climate are the result of an increase in the world's temperature, a 
ll about 1945, cooled until about 1975 and have risen steadily to present day. There are several possible reasons for the warming, scientists

# 8.Citation per sentence

Se implementó citation per sentence, donde cada oración de la respuesta es vinculada con el chunk más relevante recuperado mediante embeddings, permitiendo rastrear la fuente de la información.

In [75]:
#Dividimos oraciones
import re

def split_sentences(text):
    sentences = re.split(r'[.!?]\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

In [76]:
#Busqueda de chunk parecido
def cite_sentences(answer, docs):

    sentences = split_sentences(answer)

    sent_emb = embedder.encode(sentences)
    doc_emb = embedder.encode(docs)

    citations = []

    for i, s_emb in enumerate(sent_emb):

        scores = np.dot(doc_emb, s_emb)

        best_doc = np.argmax(scores)

        citations.append(f"{sentences[i]} [Doc{best_doc+1}]")

    return "\n".join(citations)

In [77]:
cited_answer = cite_sentences(answer, reranked_docs)

print(cited_answer)

system
Answer ONLY using provided context [Doc2]
user
Context:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming [Doc1]
Many global warming experts say the phenomenon, if unchecked, is capable of altering the world's climate and geography [Doc1]
In the worst-case scenario, experts say oceans could rise to overwhelming and catastrophic levels, flooding cities and altering seashores [Doc1]
Other
 be significant because it was going to be an example of climate change." "The other thing was that it meant it was really happening [Doc2]
It wasn't a joke [Doc2]
It wasn't just statistics [Doc2]
It was really happening." He calls his discovery Warming Island [Doc2]
Many climatologists and scientists say arctic ice melt and other changes in the Earth's climate are the result of an increase in the world's temperature, a 
ll about 1945, cooled until about 1975 and have risen steadily to present day [Doc3]
There ar

# 9.Hallucination guard + grounding score (0–1)

In [78]:
def hallucination_guard(answer, context):
    hits = sum(1 for s in answer.split() if s.lower() in context.lower())
    return hits / max(len(answer.split()),1)

 Funcion de Grounding Score

El **grounding score** mide qué proporción de la respuesta del modelo proviene del contexto proporcionado.

**Escala:**

- **0** → Alucinación total (la respuesta no se basa en los documentos)
- **1** → Completamente basado en los documentos

In [79]:
def grounding(answer, retrieved_chunks):
    ctx = " ".join(retrieved_chunks)
    return hallucination_guard(answer, ctx)


Consulta (Query)

In [80]:
query = "What causes climate change?"

# 1️.Retrieval (FAISS)
#docs = retrieve(query, k=10)
docs = hybrid_retrieve(query, k=10)

# 2️.Reranking
reranked_docs = rerank(query, docs, top_k=5)

# 3️.Contexto para el LLM
context = "\n".join(reranked_docs)

# 4.Generación con el LLM
answer = ask_llm(context, query)

# 5️.Citation per sentence
cited_answer = cite_sentences(answer, reranked_docs)

# 6️.Grounding score
score = grounding(answer, reranked_docs)

print("ANSWER:\n", cited_answer)
print("\nGrounding score:", score)

ANSWER:
 system
Answer ONLY using provided context [Doc5]
user
Context:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming [Doc1]
Many global warming experts say the phenomenon, if unchecked, is capable of altering the world's climate and geography [Doc1]
In the worst-case scenario, experts say oceans could rise to overwhelming and catastrophic levels, flooding cities and altering seashores [Doc1]
Other
 be significant because it was going to be an example of climate change." "The other thing was that it meant it was really happening [Doc2]
It wasn't a joke [Doc2]
It wasn't just statistics [Doc2]
It was really happening." He calls his discovery Warming Island [Doc2]
Many climatologists and scientists say arctic ice melt and other changes in the Earth's climate are the result of an increase in the world's temperature, a 
ll about 1945, cooled until about 1975 and have risen steadily to present day [Doc3]

Un **Grounding score de 0.962099** indica que la respuesta del modelo está **muy fuertemente basada en el contexto o documentos proporcionados**. En términos prácticos, significa que aproximadamente **el 96% de la información utilizada proviene del contexto**, por lo que existe **muy baja probabilidad de alucinación** y la respuesta está **bien fundamentada en los documentos recuperados**.

# 10.Evaluación Automática

In [81]:
!pip install rouge-score

In [82]:
#Importar ROUGE
from rouge_score import rouge_scorer

In [83]:
#Creamos algunas preguntas de evalución
evaluation_set = {
    "What causes climate change?":
        "Climate change is mainly caused by greenhouse gases in the atmosphere.",

    "What are greenhouse gases?":
        "Greenhouse gases trap heat in the Earth's atmosphere.",

    "Why is global warming dangerous?":
        "Global warming can increase sea levels and extreme weather."
}

In [84]:
#Creamos el evaluador
scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)

In [85]:
query = "What causes climate change?"

In [92]:
for query, reference_answer in evaluation_set.items():

    # RETRIEVE
    #docs = retrieve(query, k=10)
    docs = hybrid_retrieve(query, k=10)

    # RERANK
    reranked_docs = rerank(query, docs, top_k=5)

    # CONTEXT
    context = "\n".join(reranked_docs)

    # LLM
    answer = ask_llm(context, query)

    # CITATION PER SENTENCE
    cited_answer = cite_sentences(answer, reranked_docs)

    # GROUNDING
    grounding_score = grounding(answer, reranked_docs)

    # ROUGE
    rouge_scores = scorer.score(reference_answer, answer)

    print("="*60)
    print("QUESTION:", query)

    print("\nANSWER:")
    print(cited_answer)

    print("\nGrounding score:", round(grounding_score,2))

    print("ROUGE-1:", round(rouge_scores['rouge1'].fmeasure,2))
    print("ROUGE-L:", round(rouge_scores['rougeL'].fmeasure,2))

QUESTION: What causes climate change?

ANSWER:
system
Answer ONLY using provided context [Doc5]
user
Context:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming [Doc1]
Many global warming experts say the phenomenon, if unchecked, is capable of altering the world's climate and geography [Doc1]
In the worst-case scenario, experts say oceans could rise to overwhelming and catastrophic levels, flooding cities and altering seashores [Doc1]
Other
 be significant because it was going to be an example of climate change." "The other thing was that it meant it was really happening [Doc2]
It wasn't a joke [Doc2]
It wasn't just statistics [Doc2]
It was really happening." He calls his discovery Warming Island [Doc2]
Many climatologists and scientists say arctic ice melt and other changes in the Earth's climate are the result of an increase in the world's temperature, a 
ll about 1945, cooled until about 1975 and hav

De los resultados:

- El sistema RAG logra alto grounding (>0.8), lo que indica que las respuestas están basadas en documentos recuperados.

- Las métricas ROUGE son bajas porque el modelo genera respuestas parafraseadas, no exactamente iguales a la referencia.

- El sistema RAG reduce alucinaciones al obligar al modelo a usar contexto recuperado.

In [91]:
# =====================================
# DEMO DEL SISTEMA RAG
# =====================================


query = "What causes climate change?"

print("USER QUESTION:")
print(query)

# 1 Retrieval híbrido
docs = hybrid_retrieve(query, k=10)

print("\nRETRIEVED DOCUMENTS:")
for i, d in enumerate(docs[:3]):
    print(f"\nDoc{i+1}:")
    print(d[:200])

# 2 Reranking
reranked_docs = rerank(query, docs, top_k=5)

# 3 Contexto final para el LLM
context = "\n".join(reranked_docs)

# 4 Generación con LLM
answer = ask_llm(context, query)

# 5 Citation per sentence
cited_answer = cite_sentences(answer, reranked_docs)

print("\nGENERATED ANSWER WITH CITATIONS:")
print(cited_answer)

# 6 Grounding score
grounding_score = grounding(answer, reranked_docs)

print("\nGrounding score:", round(grounding_score,2))

USER QUESTION:
What causes climate change?

RETRIEVED DOCUMENTS:

Doc1:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming. Many global warming experts say the phenomenon, if unchecked, is

Doc2:
ts. "For the last 30 years, there's no way there's anything natural that can explain it," Stephen Schneider, a professor of environmental studies at Stanford University in California, said. "A vast bu

Doc3:
acted as though the rules -- even those of geography and climate -- did not apply to him. But when you're Wright, you're right. In 1935, department store magnate Stanley Marcus (of Neiman-Marcus fame)

GENERATED ANSWER WITH CITATIONS:
system
Answer ONLY using provided context [Doc5]
user
Context:
nd other changes in the Earth's climate are the result of an increase in the world's temperature, a trend widely called global warming [Doc1]
Many global warming experts say the phenomenon, if unchecked, is capable